In [ ]:
!pip install -q qdrant-client sentence-transformers requests groq

In [ ]:
import os
import requests
github_url =  "https://github.com/LavanyaKommera/ai-engineer-portfolio/blob/main/Vitaminsandmineralstxt.txt"

def load_document(url: str) -> str:
    """Fetch a plain-text file from a raw GitHub URL."""
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.text

raw_text = load_document(github_url)
print(f"Loaded {len(raw_text):,} characters")
print(raw_text[:400])  # Sanity check

In [ ]:
CHUNK_SIZE = 100

def parse_word_chunks(text: str, chunk_size: int = CHUNK_SIZE) -> list[dict]:
    # Strip markdown heading symbols and blank lines
    clean_lines = []
    for line in text.splitlines():
        line = line.strip().lstrip("#").strip()
        if line:
            clean_lines.append(line)

    # Join everything into one word list and slice
    words = " ".join(clean_lines).split()

    chunks = []
    for i in range(0, len(words), chunk_size):
        content = " ".join(words[i : i + chunk_size])
        chunks.append({
            "chunk_index": len(chunks),
            "content": content,
        })

    return chunks

In [ ]:
chunks = parse_word_chunks(raw_text)
print(f"Total chunks: {len(chunks)}")

In [ ]:
for chunk in chunks[:5]:
    print("─" * 55)
    print(f"Content : {chunk['content'][:200]}…")

In [ ]:
def build_chunk_text(chunk: dict) -> str:
    return chunk["content"]

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

In [ ]:
# Extract Chunk Texts
chunk_texts = [build_chunk_text(c) for c in chunks]

print(f"Embedding {len(chunk_texts)} chunks …")
embeddings = embedder.encode(chunk_texts, show_progress_bar=True)

print(f"Shape: {embeddings.shape}")

In [ ]:
!pip install chromadb

In [ ]:
import chromadb
client = chromadb.PersistentClient("./tmp/chromadb")

COLLECTION_NAME = "vitamindocs"
DIM = embedder.get_embedding_dimension()

client.delete_collection("vitaminkb_collection")

collection = client.get_or_create_collection(
    name="vitaminkb_collection",
)
print("Collection created.", collection.name)

In [ ]:
# Creating Points

#start ID sequence
ids = [f"chunk_{i}" for i in range(len(chunk_texts))]

metadatas = [
    {
        "chunk_index": i,
        "source": "insurance_rag_knowledge_base.txt"
    }
    for i in range(len(chunk_texts))
]

if collection.count() == 0:
    collection.add(
        ids=ids,
        documents=chunk_texts,
        embeddings=embeddings.tolist(),
        metadatas=metadatas
    )
    print(f"Inserted {len(chunk_texts)} chunks")
else:
    print(f"Collection already has {collection.count()} records — skipping insert")


In [ ]:
print(f"collection count: {collection.count()}")

sample = collection.get(limit=1, include=["embeddings"])
print(f"Dimensions : {len(sample['embeddings'][0])}")

In [ ]:
def retrieve(
    query: str,
    top_k: int = 5
) -> list[dict]:
    """
    Embed the query and return the top-k most similar chunks.

    Args:
        query          : User's question.
        top_k          : Number of chunks to return.
        section_filter : Optional H2 heading to restrict the search scope.
    """
    query_vector = embedder.encode(query).tolist()

    hits = collection.query(
        query_embeddings=query_vector,
        n_results=top_k,
        include=["documents","metadatas","distances"]
    )
    retrieved_chunks = []
    if hits and hits['documents'] and len(hits['documents']) > 0:
        for i in range(len(hits['documents'][0])):
            chunk = {
                "content": hits['documents'][0][i],
                "score": round(hits['distances'][0][i], 4),
                **hits['metadatas'][0][i]
            }
            retrieved_chunks.append(chunk)

    return retrieved_chunks

In [ ]:
results = retrieve("What is the source of Riboflavin? ")
for r in results:
    print(f"[score={r['score']}]")
    print(f"  {r['content'][:200]}…\n")

In [ ]:
SYSTEM_PROMPT = """You are a helpful HR assistant.
Answer the user's question using ONLY the context provided below.
If the context does not contain enough information, say so — do not make things up.
Always cite the section name when referencing specific information."""

In [ ]:
def build_context(retrieved_chunks: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(f"[Source {i}]\n{chunk['content']}")
    return "\n\n---\n\n".join(parts)

In [ ]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [ ]:
from groq import Groq

groq_client = Groq()   # Reads GROQ_API_KEY from environment automatically
GROQ_MODEL  = "openai/gpt-oss-safeguard-20b"

def rag(query: str, top_k: int = 5):
    """
    End-to-end RAG pipeline:
      1. Retrieve relevant chunks from Qdrant
      2. Format them as a context block
      3. Send context + query to Groq and return the answer
    """
    # Step 1 — Retrieve
    chunks = retrieve(query, top_k=top_k)
    if not chunks:
        return "No relevant content found in the document."

    # Step 2 — Build context
    context = build_context(chunks)

    # Step 3 — Generate
    user_message = f"Context:\n{context}\n\nQuestion: {query}"

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.2,   # Low = factual;  High = creative
    )
    return response.choices[0].message.content, context

In [ ]:
answer, context = rag("Why is Riboflavin important")
print(answer)
print(f"{250*'='}")
print(f"\n\nSOURCES:\n {context}")